# DDQN: Double Deep Q-Network

**Paper**: van Hasselt et al., 2015 — *Deep Reinforcement Learning with Double Q-learning*

## The Problem: Q-value Overestimation

DQN computes the TD target as:
$$y = r + \gamma \max_{a'} Q_{\text{target}}(s', a')$$

The **same network** both *selects* the best action and *evaluates* its value. When Q-values have estimation noise, `max` always picks the overestimated action — causing systematic upward bias that compounds over training.

## The Fix: Decouple Selection and Evaluation

Use the **online network** to select the action, but the **target network** to evaluate it:

$$y^{\text{DDQN}} = r + \gamma \; Q_{\text{target}}\!\left(s',\; \arg\max_{a'} Q_{\text{online}}(s', a')\right)$$

This one-line change significantly reduces overestimation. Everything else (replay buffer, target network, epsilon-greedy) stays identical to DQN.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import gymnasium as gym
from collections import deque
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )
    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity=10_000):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a, r, ns, done):
        self.buf.append((s, a, r, ns, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            torch.FloatTensor(np.array(s)).to(device),
            torch.LongTensor(a).to(device),
            torch.FloatTensor(r).to(device),
            torch.FloatTensor(np.array(ns)).to(device),
            torch.FloatTensor(d).to(device),
        )

    def __len__(self):
        return len(self.buf)

## DDQN Agent

The only difference from DQN is in `update()`: the best next action is **chosen** by `online_net`, but its **value** is read from `target_net`.

In [ ]:
class DDQNAgent:
    def __init__(self, state_dim, action_dim,
                 lr=1e-3, gamma=0.99, batch_size=64,
                 buffer_size=10_000, target_update=100,
                 eps_start=1.0, eps_end=0.01, eps_decay=0.995):

        self.action_dim    = action_dim
        self.gamma         = gamma
        self.batch_size    = batch_size
        self.target_update = target_update
        self.eps           = eps_start
        self.eps_end       = eps_end
        self.eps_decay     = eps_decay
        self.step_count    = 0

        self.online_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = torch.optim.Adam(self.online_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)

    def select_action(self, state):
        if random.random() < self.eps:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(device)
            return self.online_net(s).argmax(1).item()

    def update(self):
        if len(self.buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)

        # Current Q(s, a)
        q_values = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            # DDQN: online selects action, target evaluates value
            best_actions = self.online_net(next_states).argmax(1, keepdim=True)  # <-- key change
            next_q       = self.target_net(next_states).gather(1, best_actions).squeeze(1)
            target       = rewards + self.gamma * next_q * (1 - dones)

        loss = F.mse_loss(q_values, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), 10.0)
        self.optimizer.step()

        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        self.step_count += 1
        if self.step_count % self.target_update == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())
        return loss.item()

## Training

In [ ]:
env = gym.make('CartPole-v1')
state_dim  = env.observation_space.shape[0]
action_dim = env.action_space.n

agent = DDQNAgent(state_dim, action_dim)

EPISODES = 400
episode_rewards = []

for ep in range(EPISODES):
    state, _ = env.reset(seed=ep)
    total_reward = 0
    while True:
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.buffer.push(state, action, reward, next_state, float(done))
        agent.update()
        state = next_state
        total_reward += reward
        if done:
            break
    episode_rewards.append(total_reward)
    if (ep + 1) % 50 == 0:
        avg = np.mean(episode_rewards[-50:])
        print(f'Ep {ep+1:4d}/{EPISODES}  Avg(50): {avg:6.1f}  eps: {agent.eps:.3f}')

env.close()

## DQN vs DDQN Comparison

Run DQN baseline with same hyperparams to see the overestimation difference.

In [ ]:
class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, batch_size=64,
                 buffer_size=10_000, target_update=100, eps_start=1.0, eps_end=0.01, eps_decay=0.995):
        self.action_dim = action_dim; self.gamma = gamma
        self.batch_size = batch_size; self.target_update = target_update
        self.eps = eps_start; self.eps_end = eps_end; self.eps_decay = eps_decay
        self.step_count = 0
        self.online_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()
        self.optimizer = torch.optim.Adam(self.online_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size)
    def select_action(self, state):
        if random.random() < self.eps:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            return self.online_net(torch.FloatTensor(state).unsqueeze(0).to(device)).argmax(1).item()
    def update(self):
        if len(self.buffer) < self.batch_size: return None
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        q_values = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1)[0]  # DQN: max from target directly
            target = rewards + self.gamma * next_q * (1 - dones)
        loss = F.mse_loss(q_values, target)
        self.optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), 10.0)
        self.optimizer.step()
        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        self.step_count += 1
        if self.step_count % self.target_update == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())
        return loss.item()

torch.manual_seed(42); np.random.seed(42); random.seed(42)
dqn_agent = DQNAgent(state_dim, action_dim)
dqn_rewards = []
env2 = gym.make('CartPole-v1')
for ep in range(EPISODES):
    state, _ = env2.reset(seed=ep)
    total = 0
    while True:
        a = dqn_agent.select_action(state)
        ns, r, term, trunc, _ = env2.step(a)
        dqn_agent.buffer.push(state, a, r, ns, float(term or trunc))
        dqn_agent.update()
        state = ns; total += r
        if term or trunc: break
    dqn_rewards.append(total)
env2.close()

def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(11, 4))
x = np.arange(len(smooth(episode_rewards))) + 19
plt.plot(x, smooth(dqn_rewards),     color='tomato',    linewidth=2, label='DQN')
plt.plot(x, smooth(episode_rewards), color='steelblue', linewidth=2, label='DDQN')
plt.axhline(195, color='gray', linestyle='--', label='Solved (195)')
plt.xlabel('Episode'); plt.ylabel('Total Reward (smoothed)')
plt.title('DQN vs DDQN on CartPole-v1')
plt.legend(); plt.tight_layout(); plt.show()

## Summary

| | DQN | DDQN |
|---|---|---|
| **Action selection** | `target_net.max(1)` | `online_net.argmax(1)` |
| **Action evaluation** | `target_net` (same network) | `target_net` (different from selector) |
| **Overestimation** | High | Significantly reduced |
| **Code change** | baseline | 1 line in `update()` |

**Key insight**: When the same noisy network selects AND evaluates via `max`, errors compound upward. DDQN uses the online net to select and the (less correlated) target net to evaluate — so one overestimating does not confirm the other's bias.